In [27]:
#Import the required packages
#Import package pandas for data analysis
import pandas as pd


### Define helping functions

In [28]:
def build_dqr_table(df, columns):
    """
    Build and save a Data Quality Report table.
    
    Parameters:
    -----------
    df : pandas.DataFrame
    columns : Index
        Features to include
    csv_filename : str
        Output filename
    
    Returns:
    --------
    pandas.DataFrame
        Complete DQR table
    """
    stats = df[columns].describe().T
    perc_missing = 100 * (df[columns].isnull().sum() / len(df))
    cardinality = df[columns].nunique()
    
    dqr_table = pd.concat([
        stats,
        pd.DataFrame(perc_missing, columns=['%missing']),
        pd.DataFrame(cardinality, columns=['cardinality'])
    ], axis=1)
    
    return dqr_table

print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


In [29]:
def load_data(filename):
    # Reading from a csv file, into a data frame with error handling
    try:
        # If the csv file has any special symbols, it helps to specify the file encoding.
        df = pd.read_csv(filename, 
                        keep_default_na=True, 
                        delimiter=',', 
                        skipinitialspace=True)
        
        print(f"✓ Successfully loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")
        
        # Show the first few rows in the data frame
        display(df.head(5))
        
    except FileNotFoundError:
        print("ERROR: The file 'ppr-group-25204989-train.csv' was not found.")
        print("Please ensure the file is in the current working directory.")
    except Exception as e:
        print(f"ERROR: An unexpected error occurred: {e}")

    return df

print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


In [30]:
# Validate that the dataframe has data and expected columns
def validate_data(df):
    print("Dataset Validation Checks:")
    print("="*60)

    # Check 1: Dataset is not empty
    if df.empty:
        print("⚠ WARNING: Dataset is empty!")
    else:
        print(f"✓ Dataset contains {df.shape[0]:,} rows")

    # Check 2: Check for duplicate rows
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        print(f"⚠ WARNING: Found {duplicate_count} duplicate rows")
    else:
        print("✓ No duplicate rows found")

    # Check 3: Check if all columns are unnamed
    unnamed_cols = [col for col in df.columns if 'Unnamed' in str(col)]
    if unnamed_cols:
        print(f"⚠ WARNING: Found {len(unnamed_cols)} unnamed columns: {unnamed_cols}")
    else:
        print("✓ All columns have names")

    # Check 4: Basic column overview
    print(f"\n✓ Dataset has {df.shape[1]} columns:")
    print(f"  - Column names: {list(df.columns)}")

print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


### 1. Load data and investigate the structe

In [31]:
df=load_data("ALF02.csv")

✓ Successfully loaded dataset with 1260 rows and 7 columns


,Statistic Label,Year,Age Group,Sex,Education Attainment Level,UNIT,VALUE
0,Annual percentage of part-time work,2019,All ages,Both sexes,Levels of Education (Levels 0-8),%,18.1
1,Annual percentage of part-time work,2019,All ages,Both sexes,Less than primary (Level 0),%,NaN
2,Annual percentage of part-time work,2019,All ages,Both sexes,Primary and lower secondary education (Levels ...,%,25.1
3,Annual percentage of part-time work,2019,All ages,Both sexes,Upper secondary and post-secondary non-tertiar...,%,22.9
4,Annual percentage of part-time work,2019,All ages,Both sexes,Tertiary education (Levels 5-8),%,13.3


In [32]:
validate_data(df)

Dataset Validation Checks:
✓ Dataset contains 1,260 rows
✓ No duplicate rows found
✓ All columns have names

✓ Dataset has 7 columns:
  - Column names: ['Statistic Label', 'Year', 'Age Group', 'Sex', 'Education Attainment Level', 'UNIT', 'VALUE']


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1260 entries, 0 to 1259
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Statistic Label             1260 non-null   object 
 1   Year                        1260 non-null   int64  
 2   Age Group                   1260 non-null   object 
 3   Sex                         1260 non-null   object 
 4   Education Attainment Level  1260 non-null   object 
 5   UNIT                        1260 non-null   object 
 6   VALUE                       589 non-null    float64
dtypes: float64(1), int64(1), object(5)
memory usage: 69.0+ KB


Value (employment rate) and year are quantitative. 
TLIST(A1) is the encode of year and is also quantitative.
Others are nominal.

### 2. Invest each column and data distribution

In [34]:

# Convert all object columns to categorical
for col in df.select_dtypes('object').columns:
    df[col] = df[col].astype('category')

df['Year']=df['Year'].astype('category')

# Update column groups after type corrections
numeric_columns = df.select_dtypes(['int64', 'float64']).columns
category_columns = df.select_dtypes('category').columns

print("✓ Data types corrected:")
print(f"  - Numeric features: {len(numeric_columns)}")
print(f"  - Categorical features: {len(category_columns)}")

✓ Data types corrected:
  - Numeric features: 1
  - Categorical features: 6


In [35]:
df_numeric_dqr = build_dqr_table(df, numeric_columns)
display(df_numeric_dqr)

,count,mean,std,min,25%,50%,75%,max,%missing,cardinality
VALUE,589.0,22.785569,12.846529,2.2,13.4,20.1,30.4,73.4,53.253968,336


There are over 50% percent missing values in the column value (Annual percentage of part-time work). Needed to be cleaned later.

In [36]:
df_categorical_dqr = build_dqr_table(df, category_columns)
display(df_categorical_dqr)

,count,unique,top,freq,%missing,cardinality
Statistic Label,1260,1,Annual percentage of part-time work,1260,0.0,1
Year,1260,7,2019,180,0.0,7
Age Group,1260,10,20 - 24 years,126,0.0,10
Sex,1260,3,Both sexes,420,0.0,3
Education Attainment Level,1260,6,Less than primary (Level 0),210,0.0,6
UNIT,1260,1,%,1260,0.0,1


According to the table,age groups has the most categories, 10 in total. It would affect the visualization design. <br>
Year comes second but it is actually quantitative. <br>
 Education Level has 6 labels, and may also need to be carefully considered when encoding.<br>
 The number of categories of other features are fine.

In [37]:
# Count each value in every categorical column, including missing values.
for col in category_columns:
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="count")
    )

    print(f"\n{col}")
    display(counts)


Statistic Label


,value,count
0,Annual percentage of part-time work,1260



Year


,value,count
0,2019,180
1,2020,180
2,2021,180
3,2022,180
4,2023,180
5,2024,180
6,2025,180



Age Group


,value,count
0,20 - 24 years,126
1,25 - 29 years,126
2,30 - 34 years,126
3,35 - 39 years,126
4,40 - 44 years,126
5,45 - 49 years,126
6,50 - 54 years,126
7,55 - 59 years,126
8,60 - 64 years,126
9,All ages,126


... (2 outputs left out)


Education Attainment Level


,value,count
0,Less than primary (Level 0),210
1,Level of education - not stated,210
2,Levels of Education (Levels 0-8),210
3,Primary and lower secondary education (Levels ...,210
4,Tertiary education (Levels 5-8),210
5,Upper secondary and post-secondary non-tertiar...,210



UNIT


,value,count
0,%,1260


In [38]:
# Count the missing rows in every attributes.
for col in category_columns:
    missing_stats = (
        df.groupby(col, observed=True, dropna=False)["VALUE"]
        .agg(
            total_rows="size",
            valid_rows="count",
        )
    )

    missing_stats["missing_rows"] = (
        missing_stats["total_rows"] - missing_stats["valid_rows"]
    )
    missing_stats["missing_pct"] = (
        missing_stats["missing_rows"]
        .div(missing_stats["total_rows"])
        .mul(100)
        .round(2)
    )

    print(f"\n{col}")
    display(missing_stats.sort_values("missing_pct", ascending=False))


Statistic Label


,total_rows,valid_rows,missing_rows,missing_pct
Statistic Label,,,,
Annual percentage of part-time work,1260,589,671,53.25



Year


,total_rows,valid_rows,missing_rows,missing_pct
Year,,,,
2025,180,69,111,61.67
2022,180,70,110,61.11
2024,180,70,110,61.11
2021,180,73,107,59.44
2023,180,74,106,58.89
2020,180,112,68,37.78
2019,180,121,59,32.78



Age Group


,total_rows,valid_rows,missing_rows,missing_pct
Age Group,,,,
25 - 29 years,126,41,85,67.46
30 - 34 years,126,42,84,66.67
35 - 39 years,126,51,75,59.52
40 - 44 years,126,55,71,56.35
20 - 24 years,126,56,70,55.56
45 - 49 years,126,58,68,53.97
50 - 54 years,126,60,66,52.38
55 - 59 years,126,63,63,50.00
60 - 64 years,126,69,57,45.24


... (2 outputs left out)


Education Attainment Level


,total_rows,valid_rows,missing_rows,missing_pct
Education Attainment Level,,,,
Less than primary (Level 0),210,0,210,100.00
Level of education - not stated,210,13,197,93.81
Primary and lower secondary education (Levels 1-2),210,71,139,66.19
Tertiary education (Levels 5-8),210,151,59,28.10
Upper secondary and post-secondary non-tertiary education (Levels 3 and 4),210,161,49,23.33
Levels of Education (Levels 0-8),210,193,17,8.10



UNIT


,total_rows,valid_rows,missing_rows,missing_pct
UNIT,,,,
%,1260,589,671,53.25


### 3. Clean the data
Drop the useless colomns and rows with missing values.

Some columns are the encoding of other columns, including STATISTIC, TLIST(A1), C02076V02508, C02199V02655, C04283V05060, C03788V04538. These columns should be deleted. <br>
Besides, Statistic Label and UNIT have only one value which indicate the sematic meaning of the data. They are not useful and can be deleted.

In [39]:
# Drop encoded columns and constant metadata that are not useful for visualization.
col_to_drop = [
    "Statistic Label", "UNIT"
]
df = df.drop(columns=col_to_drop)

print("Columns dropped successfully: ")
print(col_to_drop)

Columns dropped successfully: 
['Statistic Label', 'UNIT']


In [40]:
category_columns = df.select_dtypes('category').columns
df_categorical_dqr = build_dqr_table(df, category_columns)
display(df_categorical_dqr)

,count,unique,top,freq,%missing,cardinality
Year,1260,7,2019,180,0.0,7
Age Group,1260,10,20 - 24 years,126,0.0,10
Sex,1260,3,Both sexes,420,0.0,3
Education Attainment Level,1260,6,Less than primary (Level 0),210,0.0,6


Rows with missing value should be deleted

In [41]:
# Remove rows with missing values in any column.
df = df.dropna()

print("Rows with missing values removed.")

Rows with missing values removed.


In Education Level, the not-stated does not represent a defined educational attainment category and cannot be meaningfully compared with other category. It should be removed.

In [42]:
# Remove rows with an unspecified educational attainment level.
col = "Education Attainment Level"
df = df.loc[df[col] != "Level of education - not stated"].copy()

print("Education Attainment Level removed")

Education Attainment Level removed


### 4. Change the labels into more readable labels for further visualization
VALUE -> Part-time Work Percentage <br>
Education Attainment Level -> Education Level <br>
NUTS 2 Region -> Region <br>
Levels of Education (Levels 0-8) -> All levels<br>
Tertiary education (Levels 5-8)	-> Tertiary (5–8)<br>
Upper secondary and post-secondary non-tertiar...	-> Upper / Post-Secondary (3–4)<br>
Primary and lower secondary education Levels ...	-> Primary / Lower Secondary (1–2)<br>
15-24 years -> 15-24

In [43]:
# Rename columns to make visualization labels easier to read.
df = df.rename(columns={
    "VALUE":"Part-time Work Percentage",
    "Education Attainment Level": "Education Level"
})

# Shorten education labels while preserving the categorical data type.
labels = {
    "Levels of Education (Levels  0-8)": "All levels",
    "Tertiary education (Levels 5-8)": "Tertiary (5-8)",
    "Upper secondary and post-secondary non-tertiary education (Levels 3 and 4)": "Upper / Post-Secondary (3–4)",
    "Primary and lower secondary education (Levels 1-2)": "Primary / Lower Secondary (1–2)",
}
df["Education Level"] = df["Education Level"].cat.rename_categories(labels)

# Remove the trailing "years" from age labels and preserve categorical values.
df["Age Group"] = df["Age Group"].cat.rename_categories(
    lambda label: label.removesuffix("years").rstrip()
)

# Refresh the column list for subsequent categorical summaries.
category_columns = df.select_dtypes("category").columns
display(df.head())


,Year,Age Group,Sex,Education Level,Part-time Work Percentage
0,2019,All ages,Both sexes,All levels,18.1
2,2019,All ages,Both sexes,Primary / Lower Secondary (1–2),25.1
3,2019,All ages,Both sexes,Upper / Post-Secondary (3–4),22.9
4,2019,All ages,Both sexes,Tertiary (5-8),13.3
6,2019,All ages,Male,All levels,8.7


In [44]:
# Remove the unused category label from subsequent frequency tables.
for col in df.select_dtypes("category").columns:
    df[col] = df[col].cat.remove_unused_categories()

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 576 entries, 0 to 1258
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   Year                       576 non-null    category
 1   Age Group                  576 non-null    category
 2   Sex                        576 non-null    category
 3   Education Level            576 non-null    category
 4   Part-time Work Percentage  576 non-null    float64 
dtypes: category(4), float64(1)
memory usage: 11.4 KB


In [45]:
# Count each value in every categorical column, including missing values.
for col in category_columns:
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="count")
    )

    print(f"\n{col}")
    display(counts)


Year


,value,count
0,2019,115
1,2020,109
2,2023,74
3,2021,71
4,2022,70
5,2024,69
6,2025,68



Age Group


,value,count
0,All ages,84
1,60 - 64,69
2,55 - 59,63
3,50 - 54,59
4,45 - 49,58
5,20 - 24,56
6,40 - 44,54
7,35 - 39,51
8,30 - 34,42
9,25 - 29,40



Sex


,value,count
0,Both sexes,238
1,Female,211
2,Male,127



Education Level


,value,count
0,All levels,193
1,Upper / Post-Secondary (3–4),161
2,Tertiary (5-8),151
3,Primary / Lower Secondary (1–2),71


### 4. Save the data

In [46]:
# Save the cleaned data without the DataFrame index.
df.to_csv("AnnualPercentageOfPart-timeWork.csv", index=False)